# 02 · SPMF FPClose — MinSup Pilot & Benchmark Configuration

> **Bản trình bày sạch để đưa lên GitHub — 22/11/2026.**  
> **Phạm vi:** phân tích **kết quả pilot đã lưu**, kiểm tra cấu hình minsup, và cung cấp runner tùy chọn; **không** tự chạy lại toàn bộ 56 cấu hình.

## Tổng quan

- **9** benchmark datasets đã chuẩn hóa, cùng nguồn dùng cho SPMF và thuật toán nhóm.
- **59** cấu hình từng được thử trên SPMF FPClose: **56 PASS + 3 TIMEOUT** (300 giây).
- `results/benchmark_config.csv` chốt 56 mức minsup để chạy benchmark chính thức.
- Các thời gian trong CSV được thu từ **pilot SPMF**, *không phải* runtime chính thức so sánh với thuật toán nhóm.
- `reference_closed_itemsets` chỉ là **số lượng closed itemsets**, không đủ chứng minh **correctness** từng itemset/support.

**Dataset công khai:** [Hugging Face](https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks)  
**Notebook lịch sử có đầy đủ các lần chạy thử:** [02_Benchmark_FPClose.ipynb](https://colab.research.google.com/drive/1zXK8l9ORBa3aN6MOAx_VCWZvc3zXuz39)  
**Drive dự án:** [FIM_Project](https://drive.google.com/drive/folders/1clzRNAL5cHWU5D1GEqFOk18k18_1GD99)

### Cách chạy

1. Chạy phần **1–6** để kiểm toán bảng đã có và tạo biểu đồ minh họa **pilot** (chỉ đọc CSV nhỏ).
2. Phần **7** cung cấp một runner tùy chọn nếu muốn kiểm tra **một cấu hình** bằng SPMF; mặc định không chạy.
3. **Không ghi đè** `benchmark_config.csv` hoặc `minsup_pilot_all.csv` trong Notebook này.

## 1 · Thiết lập môi trường (đọc dữ liệu có sẵn)

Chỉ mount Drive một lần, không cài Java hoặc tải lại dataset khi chưa thực sự cần. `SPMF_JAR` không cần cho các phần báo cáo từ CSV.

In [ ]:
from pathlib import Path
import os
import math
from decimal import Decimal, ROUND_CEILING
import pandas as pd
import matplotlib.pyplot as plt

if 'COLAB_GPU' in os.environ or Path('/content').exists():
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FIM_Project')
else:
    ROOT = Path(os.environ.get('FIM_PROJECT_ROOT', './FIM_Project')).expanduser()

DATA_DIR = ROOT/'data'/'benchmark'
RESULTS_DIR = ROOT/'results'
CONFIG = RESULTS_DIR/'benchmark_config.csv'
PILOT_LOG = RESULTS_DIR/'minsup_pilot_all.csv'
GRID_SUMMARY = RESULTS_DIR/'pilot_grid_summary.csv'
INVENTORY = RESULTS_DIR/'benchmark_inventory.csv'
for file in [CONFIG, PILOT_LOG, GRID_SUMMARY, INVENTORY]:
    if not file.is_file():
        raise FileNotFoundError(file)
print('Project:', ROOT)
print('Mode: read-only review of existing SPMF pilots')

## 2 · Kiểm tra schema và số cấu hình

`benchmark_config.csv`: 56 mức **đã chọn**, tất cả PASS.  
`minsup_pilot_all.csv`: lịch sử **59 lần thử** (giữ lại TIMEOUT, không chuyển các ô trống thành 0).  
`pilot_grid_summary.csv`: một dòng cho mỗi dataset.

In [ ]:
config = pd.read_csv(CONFIG)
pilot = pd.read_csv(PILOT_LOG)
grid = pd.read_csv(GRID_SUMMARY)
inventory = pd.read_csv(INVENTORY)

config_required={'dataset','n_transactions','minsup_pct','minsup_count',
                 'reference_closed_itemsets','pilot_wall_time_s',
                 'pilot_output_mb','reference_algorithm','status'}
pilot_required={'dataset','n_transactions','minsup_pct','minsup_count',
                'n_closed_itemsets','wall_time_s','output_mb','status','error'}
assert config_required.issubset(config.columns), config_required-set(config.columns)
assert pilot_required.issubset(pilot.columns), pilot_required-set(pilot.columns)
assert len(config)==56 and len(pilot)==59 and len(grid)==9
assert config['dataset'].nunique()==pilot['dataset'].nunique()==9
assert config['status'].eq('PASS').all()
assert (pilot['status']=='PASS').sum()==56
assert (pilot['status']=='TIMEOUT').sum()==3
assert not config.duplicated(['dataset','minsup_pct']).any()
assert not pilot.duplicated(['dataset','minsup_pct']).any()
assert pilot.loc[pilot['status']=='TIMEOUT','n_closed_itemsets'].isna().all()

print('Datasets:', config.dataset.nunique())
print('Selected configurations:',len(config))
print('Pilot attempts:',len(pilot))
print('Passed:',(pilot.status=='PASS').sum(),
      'Timeout:',(pilot.status=='TIMEOUT').sum())

## 3 · Đối chiếu minsup, support threshold và số lượng output

SPMF dùng ngưỡng support tuyệt đối `ceil(n_transactions × minsup_pct / 100)`; tính bằng `Decimal` để tránh sai lệch dấu phẩy động. Đối chiếu 56 cấu hình đã chọn với lịch sử pilot. Đây là kiểm tra **tính nhất quán của hồ sơ**, không so sánh với implementation của nhóm.

In [ ]:
def support_threshold(n_transactions, minsup_pct):
    return int((Decimal(str(int(n_transactions))) *
                Decimal(str(minsup_pct)) / Decimal('100'))
               .to_integral_value(rounding=ROUND_CEILING))

expected=[support_threshold(n,p) for n,p in zip(config.n_transactions,config.minsup_pct)]
assert (config['minsup_count'].to_numpy()==expected).all(), 'Sai ngưỡng support'

passed=pilot[pilot['status']=='PASS']
joined=config.merge(passed, on=['dataset','minsup_pct'], suffixes=('_config','_log'),
                   validate='one_to_one')
assert len(joined)==56
assert (joined['minsup_count_config']==joined['minsup_count_log']).all()
assert (joined['reference_closed_itemsets']==joined['n_closed_itemsets']).all()
assert (joined['n_transactions_config']==joined['n_transactions_log']).all()
assert set(config['dataset'])==set(inventory['dataset'])
print('PASS: 56/56 configurations consistent with pilot log and support thresholds.')

## 4 · Lưới minsup đã chốt

Đây là cấu hình chuẩn để các thành viên dùng **cùng input và cùng ngưỡng** cho SPMF FPClose và implementation của nhóm. Các mức bị loại vẫn nằm trong `minsup_pilot_all.csv`.

In [ ]:
levels=(config.sort_values(['dataset','minsup_pct'],ascending=[True,False])
        .groupby('dataset',sort=True)['minsup_pct']
        .agg(lambda x: ', '.join(f'{v:g}%' for v in x)))
summary=(grid.set_index('dataset')
         .join(levels.rename('selected_minsup'))
         .reset_index())
display(summary[['dataset','n_transactions','n_levels','selected_minsup',
                 'max_closed_itemsets']])

## 5 · Ba mức bị loại do TIMEOUT

Không suy diễn rằng output = 0; với TIMEOUT **chưa biết** số lượng closed itemsets cuối cùng. Giới hạn 300 giây là giới hạn đã áp dụng trong pilot, **không phải** chứng minh thuật toán sẽ không hoàn tất ở môi trường khác.

In [ ]:
failures=pilot.loc[pilot.status!='PASS',
                   ['dataset','minsup_pct','wall_time_s','status','error']]
display(failures)
assert {(r.dataset,float(r.minsup_pct)) for r in failures.itertuples(index=False)} == {
    ('kosarak',0.15),('kosarak',0.1),('T40I10D100K',0.5)}

## 6 · Biểu đồ minh họa kết quả **SPMF pilot**

Biểu đồ chỉ biểu diễn **một implementation (SPMF)**. Thời gian trong pilot chịu ảnh hưởng runtime và môi trường, không dùng làm bằng chứng kết luận FPClose nhóm nhanh/chậm hơn. Muốn xem tập khác, đổi `SELECTED_DATASET`; mặc định Retail.

In [ ]:
SELECTED_DATASET = 'retail'  # chess, mushrooms, retail, T10I4D100K, ...
view = config[config['dataset']==SELECTED_DATASET].sort_values('minsup_pct')
if view.empty:
    raise ValueError(f'Unknown dataset: {SELECTED_DATASET}')

plt.figure(figsize=(7,4))
plt.plot(view['minsup_pct'],view['reference_closed_itemsets'],marker='o')
plt.xlabel('MinSup (%)')
plt.ylabel('Number of closed itemsets')
plt.title(f'SPMF pilot — output size — {SELECTED_DATASET}')
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(view['minsup_pct'],view['pilot_wall_time_s'],marker='o')
plt.xlabel('MinSup (%)')
plt.ylabel('Pilot wall time (s)')
plt.title(f'SPMF pilot — runtime (NOT official comparison) — {SELECTED_DATASET}')
plt.grid(alpha=0.3)
plt.show()

## 7 · Tùy chọn: chạy lại **một** cấu hình SPMF FPClose

**Không chạy mặc định.** Nếu muốn kiểm chứng độc lập một trường hợp:

1. Chuẩn bị Java tương thích với `spmf.jar` (phiên thử trước phải dùng Java 25).
2. Cung cấp đường dẫn `SPMF_JAR_PATH` và `JAVA_EXECUTABLE` thực tế.
3. Bật `RUN_ONE_SPMF_CASE=True` và chọn mức cao, ví dụ `chess 90%`.
4. Output và CSV mới chỉ ghi vào thư mục **tạm `/content/fim_pilot_optional/`**; không sửa kết quả pilot lịch sử trên Drive.

Runner đếm **closed itemsets** có `#SUP:` và kiểm tra support. Việc so **từng itemset với implementation nhóm** phải dùng bộ công cụ correctness riêng của nhóm ở giai đoạn đánh giá chính thức.

In [ ]:
RUN_ONE_SPMF_CASE = False
SPMF_JAR_PATH = Path('/content/spmf.jar')    # Đổi theo file .jar thực tế
JAVA_EXECUTABLE = 'java'                     # Hoặc đường dẫn Java 25 của phiên Colab
CASE_DATASET = 'chess'
CASE_MINSUP_PCT = 90
CASE_TIMEOUT_S = 300

import shutil
import subprocess
import time
import tempfile


def run_spmf_once(dataset, minsup_pct, jar_path, java_executable='java', timeout_s=300):
    row = config[(config.dataset==dataset)&
                 (config.minsup_pct.round(8)==round(float(minsup_pct),8))]
    if len(row)!=1:
        raise ValueError('Phải chọn một cặp dataset/minsup trong benchmark_config.csv')
    row=row.iloc[0]
    source=DATA_DIR/f'{dataset}.txt'
    if not source.is_file() or not Path(jar_path).is_file():
        raise FileNotFoundError(f'Dataset or SPMF .jar unavailable: {source}, {jar_path}')
    with tempfile.TemporaryDirectory(prefix='fim_spmf_') as scratch:
        scratch=Path(scratch)
        target=scratch/f'{dataset}.txt'
        shutil.copy2(source,target)
        out=scratch/'closed_itemsets.txt'
        cmd=[str(java_executable),'-Xmx2g','-jar',str(jar_path),'run','FPClose',
             str(target),str(out),f'{float(minsup_pct):g}%']
        started=time.perf_counter()
        try:
            process=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout_s)
            elapsed=time.perf_counter()-started
        except subprocess.TimeoutExpired:
            return {'dataset':dataset,'minsup_pct':float(minsup_pct),
                    'status':'TIMEOUT','wall_time_s':timeout_s,
                    'n_closed_itemsets':None,'error':'Exceeded timeout'}
        if process.returncode!=0 or not out.is_file():
            return {'dataset':dataset,'minsup_pct':float(minsup_pct),
                    'status':'FAILED','wall_time_s':round(elapsed,3),
                    'n_closed_itemsets':None,'error':process.stderr[-800:]}
        count=0
        with out.open('r',encoding='utf-8') as fi:
            for line in fi:
                value=line.strip()
                if not value or value.startswith(('@','#','%')):
                    continue
                if '#SUP:' not in value:
                    raise ValueError(f'Unexpected SPMF output: {value[:120]}')
                lhs,rhs=value.rsplit('#SUP:',1)
                items=[int(x) for x in lhs.split()]
                support=int(rhs.strip())
                if len(items)!=len(set(items)) or support<int(row.minsup_count):
                    raise ValueError(f'Invalid itemset/support: {value[:120]}')
                count+=1
        return {'dataset':dataset,'minsup_pct':float(minsup_pct),
                'status':'PASS','wall_time_s':round(elapsed,3),
                'n_closed_itemsets':count,'error':''}

if RUN_ONE_SPMF_CASE:
    record=run_spmf_once(CASE_DATASET,CASE_MINSUP_PCT,SPMF_JAR_PATH,
                         JAVA_EXECUTABLE,CASE_TIMEOUT_S)
    print(record)
    target=Path('/content/fim_pilot_optional')
    target.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([record]).to_csv(target/'one_case.csv',index=False)
else:
    print('SKIP: optional SPMF run. Existing 56 PASS records are reused.')

## 8 · Handoff: dữ liệu đã sẵn sàng, evaluation chưa hoàn thành

### File bàn giao

| File | Nội dung |
|---|---|
| `results/benchmark_config.csv` | 56 cấu hình được chọn, n transactions, minsup, số closed itemsets SPMF pilot |
| `results/minsup_pilot_all.csv` | 59 thử nghiệm (56 PASS, 3 TIMEOUT) |
| `results/pilot_grid_summary.csv` | 9 dòng tóm tắt minsup |
| `results/benchmark_inventory.csv` | 9 dataset, số giao dịch và SHA-256 |
| `data/benchmark/MANIFEST.md` | nguồn gốc, chuẩn hóa và checksum |
| Hugging Face dataset | 9 file public đã được xác minh hash trước đó |

### Phần **chưa làm** và không được ghi sai là hoàn thành

- Correctness của FPClose nhóm: so từng **itemset + support**, không chỉ so số lượng.
- Benchmark chính thức với `ours_base` / `ours_opt` / `spmf`, chế độ `closed` / `all_fi` (nếu có), 3 lần chạy và thiết bị đồng nhất.
- Peak RAM, runtime vs. minsup, output-size, Retail scalability, transaction-length study.
- Phân tích độ phức tạp, lợi ích tối ưu và ít nhất hai hướng tối ưu mới.

**Lưu ý khoa học:** FPClose trực tiếp tạo *frequent closed itemsets*. Nếu môn yêu cầu *all frequent itemsets*, nhóm phải giải thích rõ cách reconstruction hoặc phạm vi so sánh với SPMF FP-Growth. Đừng dùng số `reference_closed_itemsets` làm số all frequent itemsets.

Bảng `benchmark_config.csv` **không phải** file `results/benchmark.csv` đo chính thức mà PDF hướng dẫn nhóm yêu cầu. Notebook sạch không tự tạo hay giả mạo số liệu chính thức.